In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # force single GPU

In [2]:
import os
import glob
import shutil

# Force single GPU (must be set before torch is imported anywhere)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("=== Checking /kaggle/input ===")
if not os.path.exists("/kaggle/input"):
    raise FileNotFoundError("/kaggle/input not found. Attach TendDataset via + Add Input.")

all_files = []
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        all_files.append(os.path.join(root, f))

if not all_files:
    raise FileNotFoundError("No files in /kaggle/input. Attach your dataset first.")

print(f"Found {len(all_files)} files in input.")

# Find project root (folder with src/ and scripts/)
project_candidates = []
for root, dirs, _ in os.walk("/kaggle/input"):
    if "src" in dirs and "scripts" in dirs:
        project_candidates.append(root)

if not project_candidates:
    fallback = "/kaggle/input/datasets/mylearningspace/tenddataset/Codegen"
    if os.path.isdir(os.path.join(fallback, "src")) and os.path.isdir(os.path.join(fallback, "scripts")):
        project_candidates = [fallback]
    else:
        raise FileNotFoundError("Could not find project folder with src/ and scripts/.")

src_project = project_candidates[0]
print("Source project path:", src_project)

# Copy to writable working directory
work_project = "/kaggle/working/project"
if os.path.exists(work_project):
    shutil.rmtree(work_project)
shutil.copytree(src_project, work_project)

os.chdir(work_project)
print("Working project root:", work_project)
print("Top-level files:", os.listdir("."))

required = ["src", "scripts", "configs", "data", "requirements.txt"]
missing = [x for x in required if not os.path.exists(x)]
if missing:
    raise FileNotFoundError(f"Missing required items: {missing}")

# Patch train_lora.py bug (missing return parser)
p = os.path.join(work_project, "scripts", "train_lora.py")
with open(p, "r", encoding="utf-8") as f:
    text = f.read()
if "return parser" not in text:
    text = text.replace(
        '        help="Disable MLflow logging.",\n    )\n\n\ndef main()',
        '        help="Disable MLflow logging.",\n    )\n    return parser\n\n\ndef main()',
    )
    with open(p, "w", encoding="utf-8") as f:
        f.write(text)
    print("Patched scripts/train_lora.py")
else:
    print("train_lora.py already patched")

# Reduce batch size for Kaggle GPU
cfg_path = os.path.join(work_project, "configs", "default.yaml")
with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = f.read()
cfg = cfg.replace("per_device_train_batch_size: 8", "per_device_train_batch_size: 2")
cfg = cfg.replace("per_device_eval_batch_size: 8", "per_device_eval_batch_size: 2")
cfg = cfg.replace("gradient_accumulation_steps: 4", "gradient_accumulation_steps: 8")
with open(cfg_path, "w", encoding="utf-8") as f:
    f.write(cfg)
print("Updated configs/default.yaml for Kaggle GPU")

print("\nCell 1 success. Ready for Cell 2.")

=== Checking /kaggle/input ===
Found 68 files in input.
Source project path: /kaggle/input/datasets/mylearningspace/tenddataset/Codegen
Working project root: /kaggle/working/project
Top-level files: ['src', 'configs', 'requirements-windows.txt', 'data', 'scripts', 'requirements.txt']
Patched scripts/train_lora.py
Updated configs/default.yaml for Kaggle GPU

Cell 1 success. Ready for Cell 2.


In [3]:
import os
import sys
from pathlib import Path

# Keep single GPU setting
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONPATH"] = os.getcwd()
sys.path.append(os.getcwd())

# Fix peft/torchao conflict on Kaggle
!pip uninstall -y torchao
!pip install -q -r requirements.txt
!pip install -q datasets transformers peft trl accelerate bitsandbytes sentencepiece huggingface_hub

# Create .env
Path(".env").write_text("""MODEL_NAME=Salesforce/codegen-350M-multi
BERTSCORE_MODEL_NAME=distilbert-base-uncased
TEND_DATASET_ID=care2achieve/tend
TEND_CACHE_DIR=/kaggle/working/data/cache/tend
MODELS_BASE_DIR=/kaggle/working/models/base
MODELS_CHECKPOINTS_DIR=/kaggle/working/models/checkpoints
RESULTS_DIR=/kaggle/working/results
""", encoding="utf-8")

print("Created .env")
print("Cell 2 success. Continue with Cell 3+")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 84.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 91.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.2 MB/s e

In [4]:
!python scripts/inspect_lora_modules.py
!python scripts/test_tend_loader.py

config.json: 1.00kB [00:00, 493kB/s]
tokenizer_config.json: 100%|████████████████████| 240/240 [00:00<00:00, 832kB/s]
tokenizer.json: 2.11MB [00:00, 7.26MB/s]
added_tokens.json: 1.00kB [00:00, 2.21MB/s]
special_tokens_map.json: 100%|████████████████| 90.0/90.0 [00:00<00:00, 383kB/s]
pytorch_model.bin: 100%|█████████████████████| 797M/797M [00:09<00:00, 83.2MB/s]
Loading weights: 100%|█| 165/165 [00:00<00:00, 3392.90it/s, Materializing param=
CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Writing model shards: 100%|███████████████████████| 1/1 [00:01<00:00,  1.12s/it]
Model: Salesforce/codegen-350M-multi
Local path: /kaggle/working/models/base/Salesforce__codegen-350M-multi


In [5]:
from pathlib import Path

p = Path("scripts/train_lora.py")
text = p.read_text(encoding="utf-8")

if "return parser" not in text:
    text = text.replace(
        '        help="Disable MLflow logging.",\n    )\n\n\ndef main()',
        '        help="Disable MLflow logging.",\n    )\n    return parser\n\n\ndef main()',
    )
    p.write_text(text, encoding="utf-8")
    print("Patched train_lora.py")
else:
    print("Already patched")

Already patched


In [10]:
!python scripts/train_lora.py --version kaggle_smoke --task text2sql --max-samples 50 --epochs 1 --device cuda:0 --no-mlflow

2026-07-09 11:07:45,665 - codegen.training - INFO - === LoRA training [text2sql (Text-to-SQL)] === model=Salesforce/codegen-350M-multi device=cuda:0 output=/kaggle/working/models/checkpoints/kaggle_smoke/text2sql
2026-07-09 11:07:45,665 - codegen - INFO - [text2sql (Text-to-SQL)] Loading training data
2026-07-09 11:07:45,749 - codegen - INFO - Loaded TEND spider/train from cache (5944 rows, /kaggle/working/data/cache/tend/care2achieve__tend/spider/train.jsonl)
2026-07-09 11:07:45,750 - codegen.training - INFO - Loaded TEND spider/train (5944 rows)
2026-07-09 11:07:45,785 - codegen - INFO - Loaded TEND bird/train from cache (2096 rows, /kaggle/working/data/cache/tend/care2achieve__tend/bird/train.jsonl)
2026-07-09 11:07:45,785 - codegen.training - INFO - Loaded TEND bird/train (2096 rows)
2026-07-09 11:07:45,785 - codegen.training - INFO - Combined TEND rows: configs=('spider', 'bird') split=train total=8040
2026-07-09 11:07:45,795 - codegen - INFO - Loaded TEND spider/test from cache (

In [14]:
from pathlib import Path

p = Path("scripts/train_all_lora.py")
text = p.read_text(encoding="utf-8")

if "def isatty(self)" not in text:
    text = text.replace(
        """        def flush(self) -> None:
            for stream in self._streams:
                stream.flush()

    stdout = sys.stdout""",
        """        def flush(self) -> None:
            for stream in self._streams:
                stream.flush()

        def isatty(self) -> bool:
            for stream in self._streams:
                if hasattr(stream, "isatty"):
                    return stream.isatty()
            return False

    stdout = sys.stdout""",
    )
    p.write_text(text, encoding="utf-8")
    print("Patched train_all_lora.py")
else:
    print("Already patched")

Already patched


In [15]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!python scripts/train_all_lora.py --version kaggle_v1 --device cuda:0 --no-mlflow

Logging to /kaggle/working/models/checkpoints/kaggle_v1/train_all_lora.log

=== LoRA training batch ===
Model: Salesforce/codegen-350M-multi
Device: cuda:0 (requested: cuda:0)
Checkpoint run: kaggle_v1
Started: 2026-07-09T11:24:08.914702+00:00

=== Training text2sql ===
2026-07-09 11:24:12,981 - codegen.training - INFO - === LoRA training [text2sql (Text-to-SQL)] === model=Salesforce/codegen-350M-multi device=cuda:0 output=/kaggle/working/models/checkpoints/kaggle_v1/text2sql
2026-07-09 11:24:12,982 - codegen - INFO - [text2sql (Text-to-SQL)] Loading training data
2026-07-09 11:24:13,070 - codegen - INFO - Loaded TEND spider/train from cache (5944 rows, /kaggle/working/data/cache/tend/care2achieve__tend/spider/train.jsonl)
2026-07-09 11:24:13,070 - codegen.training - INFO - Loaded TEND spider/train (5944 rows)
2026-07-09 11:24:13,105 - codegen - INFO - Loaded TEND bird/train from cache (2096 rows, /kaggle/working/data/cache/tend/care2achieve__tend/bird/train.jsonl)
2026-07-09 11:24:13,

In [16]:
import shutil, os

src = "/kaggle/working/models/checkpoints/kaggle_v1"
out = "/kaggle/working/kaggle_v1_adapters"

shutil.make_archive(out, "zip", src)
print("Download this file from Kaggle Output panel:")
print(out + ".zip")

# quick check
for task in ["text2sql", "sql2nosql", "nosql2doc"]:
    p = os.path.join(src, task)
    files = os.listdir(p)
    print(f"{task}: {files}")

Download this file from Kaggle Output panel:
/kaggle/working/kaggle_v1_adapters.zip
text2sql: ['README.md', 'run_metadata.json', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'checkpoint-1006', 'training_args.bin', 'tokenizer_config.json']
sql2nosql: ['README.md', 'run_metadata.json', 'adapter_model.safetensors', 'checkpoint-1500', 'adapter_config.json', 'tokenizer.json', 'training_args.bin', 'tokenizer_config.json']
nosql2doc: ['README.md', 'run_metadata.json', 'adapter_model.safetensors', 'checkpoint-1509', 'adapter_config.json', 'tokenizer.json', 'training_args.bin', 'tokenizer_config.json']
